# Vision-Language Model with ViT and LLaMA

<img src="https://i.postimg.cc/NFbvHbfW/Chat-GPT-Image-19-sept-2026-14-39-53.png">



The architecture consists of four main components:

* Vision Transformer (ViT): extracts meaningful visual features from an input image.
* Projector: maps the visual features into the embedding space expected by the language model.
* Text Encoder / Tokenizer: converts the user's text instruction into a representation that can be processed by the language model.
* LLaMA: combines the visual and textual representations and generates a natural-language response.

The overall pipeline is:

Image → ViT → Projector → Image Embeddings


Text → Tokenization → Text Embeddings


Image + Text Embeddings → LLaMA → Text Output

Throughout the notebook, we will explore how these components interact and how visual information can be integrated into a large language model to create a multimodal system.

The final goal is to build a model capable of answering questions about images, such as:

Image: 🐱

Question: "What animal is in this image?"

Answer: "The image shows a cat."

## 1. Download the Dataset

In this project, we use the **Tunisian Landmarks Captions** dataset from Hugging Face.

The dataset contains images of Tunisian landmarks along with their corresponding textual captions. It will be used to train and evaluate our Vision-Language Model (VLM).

The dataset allows the model to learn the relationship between:

- **Visual information** → images of Tunisian landmarks
- **Textual information** → captions describing the images

The following command clones the dataset repository from Hugging Face into the Google Colab environment.

In [ ]:
# Clone the Tunisian Landmarks Captions dataset
# from Hugging Face into the Colab environment.

!git clone https://huggingface.co/datasets/firastlili/tunisian-landmarks-captions

Cloning into 'tunisian-landmarks-captions'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 33 (delta 6), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (33/33), 10.95 KiB | 3.65 MiB/s, done.
Resolving deltas: 100% (6/6), done.


### Enabling Git LFS to Download the Actual Images

The repository uses Git LFS to store images. Therefore, we need to install the Git LFS extension and retrieve the actual image files.

In [ ]:
# 1. Install the Git Large File Storage (Git LFS) tool
!apt-get update && apt-get install git-lfs -y
!git lfs install

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [359 B]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:7 http://archive.ubuntu.com/ubuntu noble InRelease
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,872 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-backports 

In [ ]:
# 2. Retrieve the actual image files stored on Hugging Face's servers
%cd /content/tunisian-landmarks-captions
!git lfs pull
%cd /content

/content/tunisian-landmarks-captions
/content


## 2. Import Required Libraries

In this section, we import the Python libraries required to build our Vision-Language Model (VLM).

The implementation uses **PyTorch** for deep learning, **Hugging Face Transformers** for the Vision Transformer (ViT) and LLaMA language model, and **PIL** for image processing.

The main libraries are used for the following purposes:

- **Base64 and io**: Handle image data and convert images between different formats.
- **Pandas**: Load and manipulate tabular data, such as image paths and captions.
- **PIL (Python Imaging Library)**: Open and process images.
- **Torchvision**: Apply image transformations before passing images to the ViT.
- **PyTorch**: Build and train the neural network components.
- **Hugging Face Transformers**: Load and configure the ViT and LLaMA models.
- **Dataset and DataLoader**: Create datasets and efficiently load training samples in batches.
- **tqdm**: Display progress bars during training and evaluation.
- **random and NumPy**: Handle random operations and numerical processing.

These libraries provide the building blocks for the complete VLM pipeline:

**Image → ViT → Projector → LLaMA → Generated Text**

In [ ]:
# Used to encode and decode image data using Base64.
import base64

# Provides tools for working with in-memory data streams.
import io

# Used to load and manipulate tabular datasets.
import pandas as pd

# Used for opening and processing images.
from PIL import Image

# Provides image preprocessing and transformation utilities.
import torchvision.transforms as transforms

# Core PyTorch library for tensors and deep learning.
import torch

# Provides neural network modules and layers.
import torch.nn as nn

# LLaMA configuration, model, and tokenizer from Hugging Face.
from transformers import LlamaConfig, LlamaForCausalLM, LlamaTokenizer

# Vision Transformer configuration and pretrained model.
from transformers import ViTConfig, ViTModel

# PyTorch utilities for creating datasets and loading data in batches.
from torch.utils.data import Dataset, DataLoader

# Displays progress bars during training and evaluation.
from tqdm import tqdm

# Python's built-in random number generation utilities.
import random

# Numerical computing library used for array operations and reproducibility.
import numpy as np

## 3. Set Random Seeds for Reproducibility

Deep learning experiments can produce slightly different results each time they are executed because some operations involve randomness.

To make our experiments more **reproducible**, we set a fixed random seed for Python, NumPy, and PyTorch.

Using the same seed helps ensure that operations such as:

- Dataset shuffling
- Random initialization
- Sampling
- Data augmentation

produce consistent results across different runs whenever possible.

We use the value **42** as the fixed seed.

The CUDA and cuDNN settings are also configured to favor deterministic behavior when training on a GPU.

In [ ]:
# Define a fixed seed to make the experiment reproducible.
SEED = 42

# Set the seed for Python's built-in random module.
random.seed(SEED)

# Set the seed for NumPy's random number generator.
np.random.seed(SEED)

# Set the seed for PyTorch's CPU random number generator.
torch.manual_seed(SEED)

# Set the seed for PyTorch operations running on CUDA GPUs.
torch.cuda.manual_seed(SEED)

# Make cuDNN operations deterministic when possible.
# This improves reproducibility but may reduce performance.
torch.backends.cudnn.deterministic = True

# Disable cuDNN's automatic algorithm benchmarking.
# This helps avoid selecting different algorithms between runs.
torch.backends.cudnn.benchmark = False

## 4. Define Model and Training Hyperparameters

In this section, we define the main **hyperparameters** used to train and configure our Vision-Language Model (VLM).

Hyperparameters control different aspects of the model architecture, image processing, and training process.

### Training Parameters

- **`BATCH_SIZE`**: Number of training samples processed in one iteration.
- **`LEARNING_RATE`**: Controls the size of the updates made to the model parameters during training.
- **`EPOCHS`**: Number of complete passes through the training dataset.
- **`EVAL_INTERVAL`**: Number of training steps between evaluation operations.

### Language Model Parameters

- **`N_EMBD`**: Dimension of the token embeddings used by the language model.
- **`N_HEAD`**: Number of attention heads in each Transformer layer.
- **`N_LAYER`**: Number of Transformer layers in the language model.
- **`DROPOUT`**: Probability used for dropout regularization.
- **`MAX_LENGTH`**: Maximum number of text tokens processed by the model.
- **`MAX_POSITION_EMBEDDINGS`**: Maximum sequence length supported by the model's positional embeddings.
- **`N_HIDDEN_LAYERS`**: Number of hidden layers specified for the model or associated architecture.

### Vision Transformer Parameters

- **`IMG_SIZE`**: Height and width to which input images are resized.
- **`PATCH_SIZE`**: Size of each image patch extracted by the Vision Transformer.
- **`IMAGE_EMBED_DIM`**: Dimension of the visual feature representation produced by the vision encoder.
- **`N_CHANNELS`**: Number of image channels. RGB images contain 3 channels.

With these parameters, an image of size **96 × 96** is divided into patches of **16 × 16** pixels before being processed by the Vision Transformer.

In [ ]:
# =========================
# Training Hyperparameters
# =========================

# Number of samples processed together in one training batch.
BATCH_SIZE = 16

# Number of hidden layers used by the corresponding model component.
N_HIDDEN_LAYERS = 16

# Maximum number of text tokens processed for each input sequence.
MAX_LENGTH = 128

# Evaluate the model after every specified number of training steps.
EVAL_INTERVAL = 10

# Learning rate used by the optimizer to update model parameters.
LEARNING_RATE = 9e-4

# Number of complete passes through the training dataset.
EPOCHS = 20


# =========================
# Language Model Parameters
# =========================

# Dimension of the token embeddings.
N_EMBD = 128

# Number of attention heads in each Transformer layer.
N_HEAD = 8

# Number of Transformer layers.
N_LAYER = 8

# Dropout probability used for regularization.
DROPOUT = 0.4


# =========================
# Vision Transformer Parameters
# =========================

# Input image size: images will be resized to 96 × 96 pixels.
IMG_SIZE = 96

# Size of each image patch: 16 × 16 pixels.
PATCH_SIZE = 16

# Dimension of the visual embeddings produced by the vision encoder.
IMAGE_EMBED_DIM = 512

# Number of image channels.
# RGB images have three channels: Red, Green, and Blue.
N_CHANNELS = 3


# =========================
# Positional Embeddings
# =========================

# Maximum number of positions supported by the language model.
MAX_POSITION_EMBEDDINGS = 128

## 5. Select the Device and Initialize the Tokenizer

In this section, we prepare the computing device and initialize the **LLaMA tokenizer**.

### Device Selection

We first check whether a CUDA-compatible GPU is available.

- If CUDA is available, the model will run on the **GPU**, which significantly accelerates training and inference.
- Otherwise, the model will run on the **CPU**.

The selected device is stored in the `device` variable and will be used later when moving tensors and models.

### LLaMA Tokenizer

We then load the tokenizer associated with the **Llama-2-7b-chat-hf** model from Hugging Face.

The tokenizer converts natural-language text into token IDs that can be processed by LLaMA.

We also configure two tokenizer settings:

- **`pad_token`**: LLaMA does not define a separate padding token by default, so we use the end-of-sequence (`EOS`) token as the padding token.
- **`padding_side = "right"`**: Padding tokens are added to the right side of the input sequence.

These settings will be useful when processing multiple text instructions together in batches.

In [ ]:
# Use the GPU if CUDA is available; otherwise, use the CPU.
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the LLaMA tokenizer from Hugging Face.
tokenizer = LlamaTokenizer.from_pretrained(
    "NousResearch/Llama-2-7b-chat-hf"
)

# LLaMA does not have a dedicated padding token,
# so we use the EOS (end-of-sequence) token for padding.
tokenizer.pad_token = tokenizer.eos_token

# Add padding tokens to the right side of the input sequence.
tokenizer.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

## 6. Load the Image Dataset

In this section, we load the image metadata and prepare the images for use in the Vision-Language Model.

The dataset directory contains:

- The image files.
- An `inputs.csv` file containing information about the images and their captions.

We first define the directory containing the dataset. We then create a helper function that reads each image and converts its binary content into a **Base64-encoded string**.

Base64 encoding allows image data to be stored directly as text. This can be useful when preparing image information for storage or transferring it together with textual data.

Next, we load `inputs.csv` into a Pandas DataFrame. Columns that contain no data are removed using `dropna()`.

Finally, we apply the image conversion function to every filename in the `file` column and store the resulting Base64 strings in a new column called `b64string_images`.

The first few rows of the DataFrame are displayed using `df.head()` to verify that the dataset has been loaded correctly.

In [ ]:
# Directory containing the dataset images and CSV metadata.
image_dir = '/content/tunisian-landmarks-captions/'


def image_file_to_base64(image_filename):
    """
    Read an image file and convert its binary content
    into a Base64-encoded string.
    """

    # Construct the complete path to the image.
    image_path = image_dir + image_filename

    # Open the image in binary read mode.
    with open(image_path, 'rb') as img_file:

        # Read the image bytes and encode them using Base64.
        b64_str = base64.b64encode(img_file.read()).decode('utf-8')

    # Return the Base64 representation of the image.
    return b64_str




In [ ]:
# Load the dataset metadata from the CSV file.
# The dataset uses ';' as the column separator.
df = pd.read_csv(
    image_dir + 'inputs.csv',
    sep=";"
)

# Remove columns that contain only missing values.
df = df.dropna(
    axis=1,
    how="all"
)

# Convert every image referenced in the 'file' column
# into a Base64-encoded string.
df['b64string_images'] = df['landmark'].apply(
    image_file_to_base64
)

# Display the first five rows to verify the loaded data.
df.head()

,landmark,caption,b64string_images
0,alleyway.jpg,A peaceful alleyway in the historic Medina of ...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAMCAgsLCgsKCg...
1,tunisianminttea.jpg,Freshly brewed Tunisian mint tea garnished wit...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAMCAgoKCgoKCg...
2,jem.jpg,"The ancient Roman Amphitheatre of El Jem, a UN...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAkGBwgHBgkIBw...
3,SidiBouSaid.jpg,"The iconic cliffside village of Sidi Bou Said,...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAkGBxMSEhUTEh...
4,medina.jpg,A narrow stone-paved street in the historic UN...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAkGBxMTEhUTEx...


## 7. Build and Test the Vision Transformer

In this section, we create the **Vision Transformer (ViT)** that will act as the visual encoder of our Vision-Language Model.

The ViT receives an image and transforms it into a numerical representation containing visual information.

### ViT Configuration

We define the architecture using `ViTConfig`:

- **`image_size`**: Input images are resized to `96 × 96` pixels.
- **`patch_size`**: Each image is divided into `16 × 16` pixel patches.
- **`num_channels`**: Images contain 3 channels because they are RGB images.
- **`hidden_size`**: Each visual token is represented using a 512-dimensional embedding.
- **`num_attention_heads`**: Each Transformer layer uses 8 attention heads.
- **`num_hidden_layers`**: The ViT contains 16 Transformer layers.
- **`intermediate_size`**: Dimension of the feed-forward layer inside each Transformer block.
- **Dropout parameters**: Used to reduce overfitting during training.

For an image of size `96 × 96` with patches of size `16 × 16`, the image is divided into:

$$
\frac{96}{16} \times \frac{96}{16} = 6 \times 6 = 36
$$

image patches.

The ViT also adds a special **[CLS] token**, resulting in:

$$
36 + 1 = 37
$$

tokens.

### Testing the ViT

We then create a batch of zero-valued images with shape:

```text
(BATCH_SIZE, N_CHANNELS, IMG_SIZE, IMG_SIZE)
= (16, 3, 96, 96)
```
The images are passed through the ViT.

The model produces a sequence of visual representations. We select the first token ([:, 0]), which corresponds to the [CLS] token.

The resulting tensor has shape:
```text
(BATCH_SIZE, IMAGE_EMBED_DIM) = (16, 512)
```

This [CLS] representation provides a compact representation of the entire image and can later be passed through the projector before being used by the language model.

The expected output is:
```text
torch.Size([16, 512])
```

In [ ]:
config = ViTConfig(
    # Size of the input image.
    image_size=IMG_SIZE,

    # Size of each image patch.
    patch_size=PATCH_SIZE,

    # Number of channels in the input image.
    # RGB images have 3 channels.
    num_channels=N_CHANNELS,

    # Dimension of the visual embeddings.
    hidden_size=IMAGE_EMBED_DIM,

    # Number of attention heads in each Transformer layer.
    num_attention_heads=N_HEAD,

    # Number of Transformer layers in the Vision Transformer.
    num_hidden_layers=N_HIDDEN_LAYERS,

    # Dimension of the intermediate feed-forward layer.
    intermediate_size=4 * IMAGE_EMBED_DIM,

    # Dropout probability applied to hidden representations.
    hidden_dropout_prob=DROPOUT,

    # Dropout probability applied to attention probabilities.
    attention_probs_dropout_prob=DROPOUT,
)

# Create the Vision Transformer using the configuration.
testvit = ViTModel(config)

# Create a dummy batch of input images.
# Shape: (BATCH_SIZE, N_CHANNELS, IMG_SIZE, IMG_SIZE)
vit_input = torch.zeros(BATCH_SIZE, N_CHANNELS, IMG_SIZE, IMG_SIZE)

# Pass the images through the Vision Transformer.
# `last_hidden_state` contains the output representations of all tokens.
# `[:, 0]` selects the [CLS] token representation.
testvit_out = testvit(vit_input).last_hidden_state[:, 0]

# Display the shape of the [CLS] token representation.
# Expected shape: (BATCH_SIZE, IMAGE_EMBED_DIM)
testvit_out.shape # (BATCH_SIZE, IMAGE_EMBED_DIM)

torch.Size([16, 512])

## 8. Build the Vision-Language Model

In this section, we define the complete **Vision-Language Model (VLM)** by combining a Vision Transformer (ViT) with a LLaMA language model.

The model contains three main components:

1. **Vision Encoder**
   - A Vision Transformer extracts visual features from the input image.
   - The `[CLS]` token is used as the image representation.

2. **Image Projector**
   - The ViT produces image embeddings with dimension `IMAGE_EMBED_DIM`.
   - The projector maps these visual embeddings to the embedding dimension expected by LLaMA (`N_EMBD`).

3. **LLaMA Language Model**
   - LLaMA receives the projected image representation together with the text embeddings.
   - It uses this combined representation to generate a natural-language response.

The overall data flow is:

```text
Image
  ↓
Vision Transformer
  ↓
[CLS] Image Embedding
  ↓
Linear Projector
  ↓
Projected Image Embedding
  ↓
        ┌─────────────────┐
        │                 │
        │  Image + Text   │
        │   Embeddings    │
        │                 │
        └────────┬────────┘
                 ↓
              LLaMA
                 ↓
          Text Prediction
```
  
### Model Initialization

The \_\_init\_\_ method creates and configures both the ViT and LLaMA components.

The Vision Transformer is configured using the image-related parameters defined previously.

The image\_projector is a linear layer that transforms the visual embedding dimension into the language-model embedding dimension:
```text
IMAGE_EMBED_DIM → N_EMBD   
```
The LLaMA configuration defines the vocabulary size, embedding dimension, number of Transformer layers, number of attention heads, and maximum sequence length.

The LLaMA model is converted to **bfloat16** to reduce memory usage and improve computational efficiency on supported hardware.

### Forward Pass

The forward() method performs the multimodal processing.

First, the input image is passed through the ViT. The \[CLS\] representation is extracted and passed through the image projector.

The text token IDs are converted into LLaMA embeddings using the model's token embedding layer.

The projected image embedding is then concatenated with the text embeddings:
```text
[Image Embedding] + [Text Embeddings]   
```
This creates the multimodal input sequence that is provided to LLaMA.

When targets are provided, the model calculates the language-modeling loss. The first target position corresponds to the image embedding, so it is assigned -100 to ignore it when calculating the loss.

When targets are not provided, the method returns only the predicted logits.

### Text Generation

The generate() method is used during inference.

It performs the same image and text embedding process but does not calculate gradients because it is decorated with @torch.no\_grad().

LLaMA then generates new tokens based on the image and text prompt.

The max\_new\_tokens parameter controls the maximum number of tokens generated as the model's response.

In [ ]:
class VisionLanguageModel(nn.Module):
    def __init__(
        self,
        n_embed,
        image_embed_dim,
        vocab_size,
        n_layer,
        n_head,
        img_size,
        patch_size,
        n_hidden_layers,
        dropout,
        pad_token_id,
        max_position_embeddings,
        n_channels,
    ):
        # Initialize the parent nn.Module class.
        super().__init__()

        # Configure the Vision Transformer.
        vit_config = ViTConfig(
            # Input image size.
            image_size=img_size,

            # Size of each image patch.
            patch_size=patch_size,

            # Number of image channels.
            # RGB images contain 3 channels.
            num_channels=n_channels,

            # Dimension of the ViT visual embeddings.
            hidden_size=image_embed_dim,

            # Number of attention heads in each ViT layer.
            num_attention_heads=n_head,

            # Number of Transformer layers in the ViT.
            num_hidden_layers=n_hidden_layers,

            # Dimension of the intermediate feed-forward layer.
            intermediate_size=4 * image_embed_dim,

            # Dropout applied to hidden representations.
            hidden_dropout_prob=dropout,

            # Dropout applied to attention probabilities.
            attention_probs_dropout_prob=dropout,
        )

        # Create the Vision Transformer.
        self.vision_encoder = ViTModel(vit_config)

        # Project the ViT image embedding into the LLaMA
        # embedding space.
        self.image_projector = nn.Linear(image_embed_dim, n_embed)

        # Configure the LLaMA language model.
        llama_config = LlamaConfig(
            # Size of the vocabulary.
            vocab_size=vocab_size,

            # Dimension of the LLaMA token embeddings.
            hidden_size=n_embed,

            # Number of Transformer layers.
            num_hidden_layers=n_layer,

            # Number of attention heads.
            num_attention_heads=n_head,

            # Maximum supported sequence length.
            max_position_embeddings=max_position_embeddings,

            # ID used for padding tokens.
            pad_token_id=int(pad_token_id),
        )

        # Create the LLaMA causal language model.
        self.llama = LlamaForCausalLM(llama_config)

        # Convert LLaMA parameters to bfloat16
        # to reduce memory usage and improve efficiency
        # on supported hardware.
        self.llama = self.llama.to(dtype=torch.bfloat16)

    def forward(self, img_array, input_ids, targets=None):
        # img_array: [BATCH_SIZE, N_CHANNELS, IMG_SIZE, IMG_SIZE]
        # input_ids: [BATCH_SIZE, MAX_LENGTH]

        # Pass the input images through the Vision Transformer.
        # The [CLS] token is selected as the image representation.
        image_embeds = self.vision_encoder(img_array).last_hidden_state[:, 0]
        # Shape: [BATCH_SIZE, IMAGE_EMBED_DIM]

        # Project the image representation into the LLaMA
        # embedding space.
        image_embeds_proj = self.image_projector(image_embeds).to(dtype=torch.bfloat16)
        # Shape: [BATCH_SIZE, N_EMBED]

        # Add a sequence dimension to the image embedding.
        image_embeds_proj = image_embeds_proj.unsqueeze(1)
        # Shape: [BATCH_SIZE, 1, N_EMBED]

        # Convert input token IDs into LLaMA token embeddings.
        text_embeds = self.llama.model.embed_tokens(input_ids).to(dtype=torch.bfloat16)
        # Shape: [BATCH_SIZE, MAX_LENGTH, N_EMBED]

        # Concatenate the image embedding with the text embeddings.
        # The image representation becomes the first element
        # of the multimodal sequence.
        input_embeds = torch.cat([image_embeds_proj, text_embeds], dim=1)
        # Shape: [BATCH_SIZE, MAX_LENGTH + 1, N_EMBED]

        # Create an attention mask for the complete multimodal sequence.
        attention_mask = torch.ones(
            input_embeds.shape[:2],
            dtype=torch.long,
            device=input_embeds.device
        )
        # Shape: [BATCH_SIZE, MAX_LENGTH + 1]

        if targets is not None:
            # targets: [BATCH_SIZE, MAX_LENGTH]

            # Add an ignored target position corresponding to
            # the image embedding.
            # -100 tells the loss function to ignore this position.
            targets = torch.cat(
                [
                    torch.full(
                        (targets.size(0), 1),
                        -100,
                        dtype=targets.dtype,
                        device=targets.device
                    ),
                    targets
                ],
                dim=1
            )
            # Shape: [BATCH_SIZE, MAX_LENGTH + 1]

            # Run LLaMA using the combined image and text embeddings.
            outputs = self.llama(
                inputs_embeds=input_embeds,
                attention_mask=attention_mask,
                labels=targets,
            )

            # Return the predicted logits and training loss.
            return outputs.logits, outputs.loss

        else:
            # Run LLaMA without target labels.
            # This mode is used when only predictions are required.
            outputs = self.llama(
                inputs_embeds=input_embeds,
                attention_mask=attention_mask,
            )

            # Return the predicted logits.
            return outputs.logits

    @torch.no_grad()
    def generate(self, img_array, input_ids, max_new_tokens=20):
        # img_array: [BATCH_SIZE, N_CHANNELS, IMG_SIZE, IMG_SIZE]
        # input_ids: [BATCH_SIZE, MAX_LENGTH]

        # Extract the [CLS] image representation from the ViT.
        image_embeds = self.vision_encoder(img_array).last_hidden_state[:, 0]

        # Project the image representation into the LLaMA
        # embedding space and add a sequence dimension.
        image_embeds_proj = self.image_projector(image_embeds).unsqueeze(1).to(
            dtype=torch.bfloat16
        )

        # Convert the text token IDs into LLaMA embeddings.
        input_embeds = self.llama.model.embed_tokens(input_ids).to(
            dtype=torch.bfloat16
        )

        # Combine the projected image embedding with the text embeddings.
        inputs_embeds = torch.cat(
            [image_embeds_proj, input_embeds],
            dim=1
        )

        # Create an attention mask for the complete multimodal sequence.
        attention_mask = torch.ones(
            inputs_embeds.shape[:2],
            dtype=torch.long,
            device=inputs_embeds.device
        )

        # Generate new tokens using LLaMA.
        generated = self.llama.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,

            # Maximum number of new tokens to generate.
            max_new_tokens=max_new_tokens,

            # Token used for padding.
            pad_token_id=self.llama.config.pad_token_id,

            # Token indicating the end of generation.
            eos_token_id=self.llama.config.eos_token_id,
        )

        # Return the generated token IDs.
        return generated

## 9. Initialize and Test the Vision-Language Model

In this section, we create an instance of the `VisionLanguageModel` using the hyperparameters defined earlier.

The model combines the Vision Transformer and LLaMA components into a single multimodal architecture.

We then move the model to the selected device, either the GPU or CPU.

To verify that the model works correctly, we create:

- A dummy image tensor with the expected image dimensions.
- A dummy sequence of token IDs with the maximum sequence length.

These dummy inputs are passed through the model to perform a test forward pass.

The output contains the **logits** predicted by LLaMA for each position in the input sequence. The expected output shape is:

```text
(BATCH_SIZE, MAX_LENGTH + 1, VOCAB_SIZE)
```
The additional +1 comes from the image embedding that is concatenated before the text embeddings.

For a single image, the output shape is therefore:
```text
(1, MAX_LENGTH + 1, tokenizer.vocab_size)   
```
This test confirms that the image encoder, image projector, text embeddings, and LLaMA language model can work together in a single forward pass.

In [ ]:
# Create an instance of the Vision-Language Model
# using the hyperparameters defined earlier.
model = VisionLanguageModel(
        N_EMBD,
        IMAGE_EMBED_DIM,
        tokenizer.vocab_size,
        N_LAYER,
        N_HEAD,
        IMG_SIZE,
        PATCH_SIZE,
        N_HIDDEN_LAYERS,
        DROPOUT,
        tokenizer.pad_token_id,
        max_position_embeddings=MAX_POSITION_EMBEDDINGS,
        n_channels=N_CHANNELS,
)

# Move the model to the selected device (GPU or CPU).
model.to(device)

# Create a dummy image tensor for testing.
# Shape: (batch_size, channels, height, width)
dummy_img = torch.randn(
    1,
    N_CHANNELS,
    IMG_SIZE,
    IMG_SIZE
).to(device)

# Create a dummy sequence of token IDs.
# Shape: (batch_size, sequence_length)
dummy_idx = torch.randint(
    0,
    tokenizer.vocab_size,
    (1, MAX_LENGTH)
).to(device)

# Perform a forward pass through the Vision-Language Model.
output = model(dummy_img, dummy_idx)

# Display the shape of the model output.
print(output.shape)

torch.Size([1, 129, 32000])


## 10. Test the Model on an Image

In this section, we test the Vision-Language Model on a real image.

First, the image is loaded and converted to **RGB** format. It is then preprocessed using the same image transformations expected by the Vision Transformer:

- Resize the image to `IMG_SIZE × IMG_SIZE`.
- Convert the image into a PyTorch tensor.
- Normalize the image using the specified mean and standard deviation values.

An additional batch dimension is added so that the image tensor has the shape:

```text
(BATCH_SIZE, N_CHANNELS, IMG_SIZE, IMG_SIZE)
```

Next, we define a text prompt:
```text
  "A photo of"   
```
The prompt is converted into token IDs using the LLaMA tokenizer.

The image tensor and tokenized prompt are then passed to the model's generate() method. The model uses the visual information from the image together with the text prompt to generate a description.

Finally, the generated token IDs are decoded back into human-readable text.

This provides an end-to-end test of the VLM:
```text
Image    
↓  
Image Preprocessing    
↓  
ViT    
↓  
Image Embedding    
↓  
Projector    
↓  
LLaMA    
↑  
Text Prompt    
↓  
Generated Text   
```

In [ ]:
# Path to the image that will be used for testing.
img_path = '/content/tunisian-landmarks-captions/ChottelDjerid.jpg'

# Open the image and convert it to RGB format.
image = Image.open(img_path).convert('RGB')

# Define the image preprocessing pipeline.
transform = transforms.Compose([
    # Resize the image to the resolution expected by the ViT.
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Convert the PIL image into a PyTorch tensor.
    transforms.ToTensor(),

    # Normalize the image using the specified mean and standard deviation.
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Apply the preprocessing transformations.
# unsqueeze(0) adds the batch dimension.
# Shape: [1, N_CHANNELS, IMG_SIZE, IMG_SIZE]
img_tensor = transform(image).unsqueeze(0).to(device)

# Define the text prompt given to the language model.
prompt = "A photo of"

# Convert the prompt into token IDs.
# The resulting tensor is moved to the selected device.
input_ids = tokenizer(
    prompt,
    return_tensors='pt'
).input_ids.to(device)

# Disable gradient calculation because we are performing inference.
with torch.no_grad():

    # Generate a textual description using the image and prompt.
    generated_ids = model.generate(
        img_tensor,
        input_ids,
        max_new_tokens=30
    )

    # Convert the generated token IDs back into human-readable text.
    generated_text = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

# Display the generated description.
print("Generated description of the given picture:")
print(generated_text)

Generated description of the given picture:
hein NegVCтина líneacomputTVotte convertedumn Tem mi стать одним experimental HTTP Claudessげ info'):љашње avoir lips racingiche article Miesun Buffer


## 11. Convert Base64 Images to Tensors

In this section, we define a helper function that converts an image stored as a **Base64-encoded string** into a PyTorch tensor suitable for the Vision Transformer.

The function performs the following steps:

1. Decode the Base64 string back into binary image data.
2. Open the image using PIL.
3. Convert the image to RGB format if necessary.
4. Resize the image to the required input size.
5. Convert the image into a PyTorch tensor.
6. Normalize the image using the same mean and standard deviation used during image preprocessing.
7. Add a batch dimension using `unsqueeze(0)`.

The returned tensor has the shape:

```text
(1, 3, img_size, img_size)
```
For the default img_size=96, the resulting shape is:
```text
(1, 3, 96, 96)
```
This function allows images stored in the DataFrame as Base64 strings to be directly converted into tensors that can be passed to the Vision-Language Model.

In [ ]:
def base64_to_tensor(base64_str, img_size=96):
    # Decode the Base64 string back into binary image data
    # and load it using PIL.
    image = Image.open(io.BytesIO(base64.b64decode(base64_str)))

    # Convert the image to RGB format if it is not already RGB.
    if image.mode != 'RGB':
        image = image.convert('RGB')

    # Define the image preprocessing pipeline.
    transform = transforms.Compose([
        # Resize the image to the required input size.
        transforms.Resize((img_size, img_size)),

        # Convert the image into a PyTorch tensor.
        transforms.ToTensor(),

        # Normalize the image using the specified mean and standard deviation.
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    # Apply the transformations and add a batch dimension.
    # Output shape: [1, 3, img_size, img_size]
    return transform(image).unsqueeze(0)

## 12. Create the Vision-Language Dataset

In this section, we define a custom PyTorch `Dataset` for our Vision-Language Model.

The `VLMDataset` class connects each image with its corresponding textual caption. The images are stored as Base64 strings in the DataFrame, while the captions are stored in the `caption` column.

For each sample, the dataset performs three main operations:

1. **Image processing**
   - Retrieves the Base64-encoded image.
   - Converts it into a PyTorch tensor using `base64_to_tensor()`.
   - Removes the temporary batch dimension because the `DataLoader` will add the batch dimension later.

2. **Caption tokenization**
   - Converts the caption into token IDs using the LLaMA tokenizer.
   - Pads or truncates the caption to `MAX_LENGTH` tokens.

3. **Target creation**
   - Creates a copy of the input token IDs.
   - Shifts the target tokens one position to the left.
   - This creates the target sequence used for **next-token prediction** during language-model training.

For example, if the input sequence is:

```text
[A, B, C, D]
```
the corresponding target sequence becomes:
```text
[B, C, D, PAD]
```
This allows the language model to learn to predict the next token given the previous tokens.

Each dataset sample therefore returns three elements:
```text
image
input_ids
targets
```
These will later be provided to the Vision-Language Model during training.

In [ ]:
class VLMDataset(Dataset):
    def __init__(self, df, img_size=96, tokenizer=None):
        # Reset the DataFrame index so that dataset indices
        # start from 0 and remain continuous.
        self.df = df.reset_index(drop=True)

        # Store the image size used for preprocessing.
        self.img_size = img_size

        # Store the tokenizer used to convert captions into token IDs.
        self.tokenizer = tokenizer

    def __len__(self):
        # Return the total number of samples in the dataset.
        return len(self.df)

    def __getitem__(self, idx):
        # Retrieve the Base64-encoded image from the DataFrame.
        img_b64 = self.df.loc[idx, 'b64string_images']

        # Retrieve the corresponding image caption.
        caption = self.df.loc[idx, 'caption']

        # Convert the Base64 image into a tensor.
        # squeeze(0) removes the batch dimension because the
        # DataLoader will create the batch dimension later.
        image = base64_to_tensor(
            img_b64,
            self.img_size
        ).squeeze(0)

        # Tokenize the caption.
        encoding = self.tokenizer(
            caption,

            # Return the result as PyTorch tensors.
            return_tensors='pt',

            # Pad all captions to MAX_LENGTH.
            padding='max_length',

            # Truncate captions longer than MAX_LENGTH.
            truncation=True,

            # Maximum number of tokens in the caption.
            max_length=MAX_LENGTH
        )

        # Remove the batch dimension added by the tokenizer.
        # Shape: [MAX_LENGTH]
        input_ids = encoding.input_ids.squeeze(0)

        # Create a copy of the input tokens to use as training targets.
        targets = input_ids.clone()

        # Shift the target sequence one position to the left.
        # This creates the next-token prediction targets.
        targets[:-1] = input_ids[1:]

        # The final target has no next token, so it is replaced
        # with the padding token.
        targets[-1] = self.tokenizer.pad_token_id

        # Return the processed image, input token IDs,
        # and target token IDs.
        return image, input_ids, targets

## 13. Prepare Training and Validation Datasets

This cell prepares the dataset for model training and validation.

- The dataframe is repeated 50 times to increase the number of training examples.
- Only the image Base64 strings and captions are kept.
- The data is split into 90% training samples and 10% validation samples.
- `VLMDataset` is used to convert the dataframe samples into tensors and tokenized captions.
- `DataLoader` creates batches for efficient training.
- The training loader shuffles the samples, while the validation loader keeps a fixed order.
- `drop_last=True` removes the final incomplete batch so that every batch has the expected batch size.
- The lengths printed at the end provide a quick check of the dataset structure and the number of returned elements from one sample.

In [ ]:
# Repeat the dataframe 50 times to increase the number of available samples.
df = pd.concat([df] * 50)[['b64string_images', 'caption']]

# Calculate the index corresponding to 90% of the dataset.
n = int(0.9 * len(df))

# Use the first 90% of the data for training.
df_train = df.iloc[:n]

# Use the remaining 10% of the data for validation.
df_val = df.iloc[n:]

# Create the training dataset using the custom VLMDataset class.
train_dataset = VLMDataset(df_train, img_size=IMG_SIZE, tokenizer=tokenizer)

# Create the validation dataset using the custom VLMDataset class.
val_dataset = VLMDataset(df_val, img_size=IMG_SIZE, tokenizer=tokenizer)

# Print the number of samples in the training dataset.
print(len(train_dataset))

# Print the number of elements returned by one dataset sample.
# Each sample contains: image, input_ids, and targets.
print(len(train_dataset[0]))

# Create the training DataLoader.
# shuffle=True randomly changes the sample order for each epoch.
# drop_last=True removes an incomplete final batch.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

# Create the validation DataLoader.
# shuffle=False keeps the validation samples in their original order.
# drop_last=True removes an incomplete final batch.
val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    drop_last=True
)

720
3


## 14. Estimate Validation Loss

This function evaluates the VLM on the validation dataset and computes the average validation loss.

- `@torch.no_grad()` disables gradient calculation because the model is only being evaluated.
- `model.eval()` switches the model to evaluation mode.
- Each validation batch contains images, input token IDs, and target token IDs.
- The data is moved to the selected device (GPU or CPU).
- The model returns logits and the loss for each batch.
- The batch losses are collected and averaged to obtain the final validation loss.

The validation loss provides a measure of how well the model performs on data that is not used for parameter updates.

In [ ]:
# Disable gradient calculation during validation to reduce memory usage
# and avoid unnecessary computation.
@torch.no_grad()
def estimate_loss(model, val_loader):
    # Store the loss from each validation batch.
    losses = []

    # Set the model to evaluation mode.
    model.eval()

    # Iterate over all batches in the validation DataLoader.
    for images, input_ids, targets in val_loader:
        # Move the images to the selected device.
        images = images.to(device)

        # Move the input token IDs to the selected device.
        input_ids = input_ids.to(device)

        # Move the target token IDs to the selected device.
        targets = targets.to(device)

        # Run the model in evaluation mode and obtain the loss.
        _, loss = model(images, input_ids, targets)

        # Store the loss value as a Python number.
        losses.append(loss.item())

    # Return the average loss across all validation batches.
    return sum(losses) / len(losses)

## 15. Train the Vision-Language Model

This function trains the Vision-Language Model using the training dataset and evaluates it on the validation dataset after each epoch.

- The Adam optimizer is created with the specified learning rate.
- The model is trained for the requested number of epochs.
- `model.train()` enables training mode.
- `tqdm` displays the training progress for each epoch.
- Images, input IDs, and target IDs are moved to the selected device.
- `optimizer.zero_grad()` clears the gradients from the previous training step.
- The model performs a forward pass and calculates the training loss.
- `loss.backward()` computes gradients through backpropagation.
- `optimizer.step()` updates the model parameters.
- The current loss is displayed periodically according to `eval_interval`.
- After each epoch, `estimate_loss()` evaluates the model on the validation set.
- The validation loss is printed to monitor the model's performance during training.

In [ ]:
# Define the training function for the Vision-Language Model.
def train_model(model, train_loader, val_loader, epochs, learning_rate, eval_interval):
    # Create the Adam optimizer with the specified learning rate.
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Iterate through the specified number of training epochs.
    for epoch in range(epochs):
        # Set the model to training mode.
        model.train()

        # Create a progress bar for the training batches.
        loop = tqdm(
            enumerate(train_loader),
            total=len(train_loader),
            desc=f"Epoch {epoch+1}/{epochs}"
        )

        # Iterate through the training batches.
        for batch_idx, (images, input_ids, targets) in loop:
            # Move the input images to the selected device.
            images = images.to(device)

            # Move the input token IDs to the selected device.
            input_ids = input_ids.to(device)

            # Move the target token IDs to the selected device.
            targets = targets.to(device)

            # Clear gradients from the previous optimization step.
            optimizer.zero_grad()

            # Perform a forward pass and compute the training loss.
            logits, loss = model(images, input_ids, targets)

            # Compute gradients using backpropagation.
            loss.backward()

            # Update the model parameters using the computed gradients.
            optimizer.step()

            # Periodically display the current training loss.
            if batch_idx % eval_interval == 0:
                loop.set_postfix(loss=loss.item())

        # Evaluate the model on the validation dataset after each epoch.
        val_loss = estimate_loss(model, val_loader)

        # Display the validation loss for the current epoch.
        print(f"Validation Loss after epoch {epoch}: {val_loss}")

## 16. Start Model Training

This cell starts the training process using the previously created model, training DataLoader, and validation DataLoader.

The function is called with the configured number of epochs, learning rate, and evaluation interval. During execution, the training progress and validation loss are displayed after each epoch.

In [ ]:
# Start training the Vision-Language Model.
# EPOCHS controls the number of training epochs.
# LEARNING_RATE controls the optimizer learning rate.
# EVAL_INTERVAL controls how often the training loss is displayed.
train_model(model, train_loader, val_loader, EPOCHS, LEARNING_RATE, EVAL_INTERVAL)

Epoch 1/20: 100%|██████████| 45/45 [00:19<00:00,  2.34it/s, loss=3.73]


Validation Loss after epoch 0: 3.43304181098938


Epoch 2/20: 100%|██████████| 45/45 [00:20<00:00,  2.23it/s, loss=2.9]


Validation Loss after epoch 1: 2.77008855342865


Epoch 3/20: 100%|██████████| 45/45 [00:17<00:00,  2.59it/s, loss=2.55]


...

Validation Loss after epoch 17: 0.08063721284270287


Epoch 19/20: 100%|██████████| 45/45 [00:18<00:00,  2.48it/s, loss=0.0788]


Validation Loss after epoch 18: 0.07897019758820534


Epoch 20/20: 100%|██████████| 45/45 [00:18<00:00,  2.48it/s, loss=0.0785]


Validation Loss after epoch 19: 0.07801835983991623


## 17. Generate a Caption for a New Image

This cell tests the trained Vision-Language Model on a new image that was not directly provided as a training batch.

- The image is loaded and converted to RGB format.
- The image is resized to the input dimensions expected by the ViT encoder.
- The image is converted into a tensor and normalized using the same preprocessing used during training.
- A batch dimension is added before moving the image to the selected device.
- The text prompt `"A photo of"` is tokenized and used as the initial text input.
- `model.eval()` switches the model to evaluation mode.
- `torch.no_grad()` disables gradient computation during inference.
- `model.generate()` generates new tokens conditioned on both the image and the text prompt.
- The generated token IDs are decoded into readable text and printed as the generated image description.

In [ ]:
# Define the path to the image that will be used for inference.
img_path = '/content/tunisian-landmarks-captions/Sbeitla.jpg'

# Open the image and convert it to RGB format.
image = Image.open(img_path).convert('RGB')

# Define the same preprocessing pipeline used for the model's image inputs.
transform = transforms.Compose([
    # Resize the image to the Vision Transformer input size.
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    # Convert the image into a PyTorch tensor.
    transforms.ToTensor(),
    # Normalize the image using the specified mean and standard deviation.
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Apply the preprocessing, add a batch dimension,
# and move the image tensor to the selected device.
img_tensor = transform(image).unsqueeze(0).to(device)

# Define the text prompt used to start the caption generation.
prompt = "A photo of"

# Tokenize the prompt and move the resulting token IDs to the selected device.
input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)

# Set the model to evaluation mode.
model.eval()

# Disable gradient computation during inference.
with torch.no_grad():
    # Generate a sequence of tokens conditioned on the image and prompt.
    generated_ids = model.generate(img_tensor, input_ids, max_new_tokens=30)

    # Convert the generated token IDs into readable text.
    generated_text = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

# Display the generated image description.
print("Generated description of the given picture:")
print(generated_text)

Generated description of the given picture:
p street theuf of of J Sa, UNCO Doug ( by arch door, for of- pitug atop to architecture-erved Roman in
